# ASL Teckenspråkstolk — Kunskapskontroll AI
**Marcus Johansson**

Det här projektet tränar en maskininlärningsmodell som känner igen amerikanskt fingerspråk (ASL) utifrån hand-landmarks detekterade av MediaPipe.

## Projektstruktur
1. Ladda och utforska data
2. Förbehandla data
3. Träna tre modeller och jämföra
4. Utvärdera bästa modellen på testdata
5. Spara modellen för Streamlit-appen

## 0. Installera paket
Kör den här cellen första gången, sen kan du kommentera bort den.

In [ ]:
# Avkommentera och kör första gången
# !pip install mediapipe opencv-python scikit-learn pandas numpy matplotlib seaborn joblib streamlit

## 1. Importera bibliotek

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import joblib

print('Alla bibliotek importerade!')

## 2. Ladda data

Ladda ned datasetet härifrån: https://www.kaggle.com/datasets/grassknoted/asl-alphabet

Alternativt ett landmark-baserat dataset (CSV): https://www.kaggle.com/datasets/debashishsau/aslamerican-sign-language-aplhabet-dataset

Lägg CSV-filen i samma mapp som den här notebooken och uppdatera filnamnet nedan.

In [ ]:
# Läs in datasetet — uppdatera filnamnet till det du laddade ned
df = pd.read_csv('asl_dataset.csv')

# Kolla hur datan ser ut
print('Storlek på datasetet:', df.shape)
print('\nFörsta raderna:')
df.head()

## 3. Utforska datan (EDA)

In [ ]:
# Kolla vilka klasser (bokstäver) som finns och hur många exempel per klass
print('Klasser i datasetet:')
print(df['label'].value_counts())

In [ ]:
# Visualisera fördelningen av klasser
plt.figure(figsize=(14, 4))
df['label'].value_counts().sort_index().plot(kind='bar', color='steelblue')
plt.title('Antal exempel per bokstav')
plt.xlabel('Bokstav')
plt.ylabel('Antal')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Kolla om det finns saknade värden
print('Saknade värden per kolumn:')
print(df.isnull().sum().sum(), 'totalt saknade värden')

## 4. Förbehandla data

In [ ]:
# Separera features (X) och labels (y)
# OBS: Uppdatera 'label' till rätt kolumnnamn om det heter något annat i ditt dataset
X = df.drop('label', axis=1).values
y = df['label'].values

print('Features shape:', X.shape)
print('Labels shape:', y.shape)

In [ ]:
# Koda om bokstäver till siffror (A=0, B=1, etc.)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print('Klasser:', le.classes_)

In [ ]:
# Dela upp i träning (80%) och test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded  # säkerställer jämn fördelning av klasser
)

print(f'Träningsdata: {X_train.shape[0]} exempel')
print(f'Testdata:     {X_test.shape[0]} exempel')

In [ ]:
# Normalisera features — viktigt för SVM och KNN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # använd SAMMA scaler som träningen

print('Data normaliserad!')

## 5. Träna och jämföra tre modeller

Vi tränar tre olika modeller och jämför dem med cross-validation på träningsdatan.
På så sätt behöver vi inget separat valideringsdataset (se teorifråga 2).

In [ ]:
# Definiera de tre modellerna
modeller = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM':           SVC(kernel='rbf', C=10, random_state=42),
    'KNN':           KNeighborsClassifier(n_neighbors=5)
}

# Kör cross-validation (5-fold) på varje modell
resultat = {}

for namn, modell in modeller.items():
    scores = cross_val_score(modell, X_train_scaled, y_train, cv=5, scoring='accuracy')
    resultat[namn] = scores
    print(f'{namn}: {scores.mean():.3f} (+/- {scores.std():.3f})')

In [ ]:
# Visualisera jämförelsen
fig, ax = plt.subplots(figsize=(8, 4))

names = list(resultat.keys())
means = [resultat[n].mean() for n in names]
stds  = [resultat[n].std()  for n in names]

bars = ax.bar(names, means, yerr=stds, capsize=5, color=['steelblue', 'coral', 'mediumseagreen'])
ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy (cross-validation)')
ax.set_title('Jämförelse av tre modeller')

for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{mean:.3f}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

## 6. Träna och utvärdera bästa modellen på testdata

Vi väljer vinnaren från jämförelsen ovan och utvärderar den på testdatan.

In [ ]:
# Träna vinnande modellen på ALL träningsdata
# Byt ut 'SVM' mot vilken modell som vann hos dig
basta_modell = modeller['SVM']
basta_modell.fit(X_train_scaled, y_train)

# Prediktera på testdata
y_pred = basta_modell.predict(X_test_scaled)

# Accuracy
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy på testdata: {acc:.3f}')

In [ ]:
# Detaljerad rapport med Precision och Recall per bokstav
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(14, 12))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — testdata')
plt.tight_layout()
plt.show()

# Vilka bokstäver förväxlar modellen?
# Diagonalen = rätt svar. Allt utanför = fel.

## 7. Spara modellen

Vi sparar modellen och scalern med joblib så att Streamlit-appen kan ladda dem.

In [ ]:
# Spara modellen och scalern
joblib.dump(basta_modell, 'asl_modell.pkl')
joblib.dump(scaler, 'asl_scaler.pkl')
joblib.dump(le, 'asl_label_encoder.pkl')

print('Modell sparad som asl_modell.pkl')
print('Scaler sparad som asl_scaler.pkl')
print('Label encoder sparad som asl_label_encoder.pkl')

## 8. Sammanfattning

Fyll i den här cellen när du är klar med projektet.

- **Bästa modell:** _(t.ex. SVM med rbf-kernel)_
- **Accuracy på testdata:** _(t.ex. 94.2%)_
- **Styrkor:** _(vad gick bra?)_
- **Begränsningar:** _(vilka bokstäver förväxlas? varför?)_
- **Nästa steg:** _(Streamlit-app med live-kamera)_